In [ ]:
import json
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import PathPatch
from matplotlib.path import Path as MplPath

from core.config import P, settings

settings.PROJECT_DIR = PROJECT_DIR
FIGURE_DIR = PROJECT_DIR / "research" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

In [ ]:
# --- Chart style -------------------------------------------------------------
# Palette validated with the dataviz palette checker (light surface #fcfcfb):
# categorical slots 1-3 clear the all-pairs CVD and normal-vision floors.
SURFACE = "#fcfcfb"
INK = "#0b0b0b"
INK_2 = "#52514e"
MUTED = "#898781"
GRID = "#e1e0d9"
BASELINE = "#c3c2b7"

SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]  # categorical slots 1-3
CRITICAL = "#d03b3b"  # status: data lost
BLUE = "#2a78d6"
BLUE_FADED = "#b7d3f6"  # sequential blue, step 150

mpl.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "savefig.dpi": 200,
        "savefig.bbox": "tight",
        "font.family": "sans-serif",
        "font.sans-serif": ["DejaVu Sans"],
        "font.size": 9,
        "text.color": INK,
        "axes.edgecolor": BASELINE,
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.titlesize": 11,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.titlepad": 24,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": False,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "xtick.labelcolor": INK_2,
        "ytick.labelcolor": INK_2,
        "xtick.major.size": 0,
        "ytick.major.size": 0,
        "legend.frameon": False,
        "legend.fontsize": 8.5,
        "lines.linewidth": 2,
    }
)


def style_value_axis(ax, axis="x"):
    """Recessive hairline grid on the value axis only; no baseline spine on it."""
    ax.grid(True, axis=axis, color=GRID, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    if axis == "x":
        ax.spines["bottom"].set_visible(False)
        ax.spines["left"].set_color(BASELINE)
    else:
        ax.spines["left"].set_visible(False)
        ax.spines["bottom"].set_color(BASELINE)


_K = 0.5523  # circular-arc control-point constant for a cubic Bezier


def _square_path(x0, y0, w, h):
    return MplPath(
        [(x0, y0), (x0 + w, y0), (x0 + w, y0 + h), (x0, y0 + h), (x0, y0)],
        [MplPath.MOVETO] + [MplPath.LINETO] * 3 + [MplPath.CLOSEPOLY],
    )


def _rounded_path(x0, y0, w, h, rx, ry, side="right"):
    """Rectangle with the two corners on `side` rounded - the mark spec's data-end.

    `rx` and `ry` are separate because the axes are in different units: one shared
    radius would stretch the arc into a spike on whichever axis has the wider range.
    """
    rx, ry = min(abs(rx), abs(w)), min(abs(ry), abs(h) / 2)
    if rx <= 0 or ry <= 0 or w == 0 or h == 0:
        return _square_path(x0, y0, w, h)
    x1, y1 = x0 + w, y0 + h

    if side == "right":
        verts = [
            (x0, y0),
            (x1 - rx, y0),
            (x1 - rx + rx * _K, y0),
            (x1, y0 + ry - ry * _K),
            (x1, y0 + ry),
            (x1, y1 - ry),
            (x1, y1 - ry + ry * _K),
            (x1 - rx + rx * _K, y1),
            (x1 - rx, y1),
            (x0, y1),
            (x0, y0),
        ]
    else:  # "top"
        verts = [
            (x0, y0),
            (x1, y0),
            (x1, y1 - ry),
            (x1, y1 - ry + ry * _K),
            (x1 - rx + rx * _K, y1),
            (x1 - rx, y1),
            (x0 + rx, y1),
            (x0 + rx - rx * _K, y1),
            (x0, y1 - ry + ry * _K),
            (x0, y1 - ry),
            (x0, y0),
        ]
    codes = [
        MplPath.MOVETO,
        MplPath.LINETO,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.LINETO,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.LINETO,
        MplPath.CLOSEPOLY,
    ]
    return MplPath(verts, codes)


def _x_per_y(ax):
    """Data-x units that render the same on-screen length as one data-y unit.

    The rounding radius is a *visual* 4px-ish arc, so it has to be converted through
    the axes aspect - otherwise a bar spanning 3,000 x-units gets a radius of 0.2 and
    renders square.
    """
    box = ax.get_position()
    width_in = ax.figure.get_figwidth() * box.width
    height_in = ax.figure.get_figheight() * box.height
    x_range = abs(np.diff(ax.get_xlim())[0])
    y_range = abs(np.diff(ax.get_ylim())[0])
    return (x_range / width_in) / (y_range / height_in)


def barh(ax, ys, widths, color, height=0.62, left=None, radius_frac=0.30, **kw):
    """Horizontal bars with a rounded data-end. `left` enables stacking."""
    ys = np.atleast_1d(np.asarray(ys, dtype=float))
    widths = np.asarray(widths, dtype=float)
    left = np.zeros_like(widths) if left is None else np.asarray(left, dtype=float)
    ry = height * radius_frac
    rx = ry * _x_per_y(ax)
    for y, w, x0 in zip(ys, widths, left):
        ax.add_patch(
            PathPatch(
                _rounded_path(x0, y - height / 2, w, height, rx, ry, "right"),
                facecolor=color,
                edgecolor="none",
                **kw,
            )
        )
    return ax


def barv(ax, xs, heights, color, width=0.68, radius_frac=0.30, **kw):
    """Vertical bars with a rounded top."""
    xs = np.atleast_1d(np.asarray(xs, dtype=float))
    heights = np.asarray(heights, dtype=float)
    rx = width * radius_frac
    ry = rx / _x_per_y(ax)
    for x, h in zip(xs, heights):
        ax.add_patch(
            PathPatch(
                _rounded_path(x - width / 2, 0, width, h, rx, ry, "top"),
                facecolor=color,
                edgecolor="none",
                **kw,
            )
        )
    return ax


DATA_DIR = FIGURE_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)


def _jsonable(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (pd.Series, pd.Index)):
        return value.tolist()
    if isinstance(value, pd.DataFrame):
        return value.to_dict(orient="list")
    raise TypeError(type(value))


def save(fig, name, data=None):
    fig.savefig(FIGURE_DIR / f"{name}.png")
    if data is not None:
        target = DATA_DIR / f"{name}.json"
        target.write_text(json.dumps(data, indent=1, ensure_ascii=False, default=_jsonable))

In [ ]:
import logging

logging.getLogger("domain.annotations").setLevel(logging.ERROR)

from data.annotations import load_annotations

SOURCE_DIR = settings.data_dir / "cleaned"

annotations = load_annotations(SOURCE_DIR)
annotations["pair"] = annotations.species + "/" + annotations.call_type

In [ ]:
# --- The experiment subset, exactly as `create_dataset.py` selects it ---------
MIN_PAIR_COUNT = 100
EXCLUDED_PAIRS = {("lw", "cc"), ("sm", "fc"), ("sb", "pcs")}  # nesting phrases
JOINED_PAIRS: dict[tuple[tuple[str, str], ...], tuple[str, str]] = {
    (("lw", "tr"), ("lw", "tj"), ("lw", "tt"), ("lw", "tf")): ("lw", "trino"),
}

for old_pairs, new_pair in JOINED_PAIRS.items():
    for old_pair in old_pairs:
        annotations.loc[
            (annotations["species"] == old_pair[0]) & (annotations["call_type"] == old_pair[1]),
            ["species", "call_type", "pair"],
        ] = new_pair + ("/".join(new_pair),)

keep = ~annotations[["species", "call_type"]].apply(tuple, axis=1).isin(EXCLUDED_PAIRS)
candidates = annotations[keep].copy()
candidates["low_freq_hz"] = candidates.low_freq_hz.clip(lower=P.f_min)

pair_counts = candidates.pair.value_counts()
CLASSES = sorted(pair_counts[pair_counts >= MIN_PAIR_COUNT].index)
experiment = candidates[candidates.pair.isin(CLASSES)].copy()

In [ ]:
counts = annotations.pair.value_counts()
selected = set(CLASSES)

fig, ax = plt.subplots(figsize=(7.6, 11.5))
ys = np.arange(len(counts))
ax.set_xlim(0, counts.max() * 1.30)
ax.set_ylim(len(counts) - 0.4, -1.0)

for y, (pair, n) in zip(ys, counts.items()):
    is_selected = pair in selected
    barh(ax, [y], [n], BLUE if is_selected else BLUE_FADED, height=0.66)
    ax.text(
        n + counts.max() * 0.014,
        y,
        f"{n:,}",
        va="center",
        ha="left",
        fontsize=7.5,
        color=INK if is_selected else MUTED,
        fontweight="bold" if is_selected else "normal",
    )

ax.axvline(MIN_PAIR_COUNT, color=MUTED, linewidth=1, linestyle=(0, (4, 3)), zorder=1)
ax.text(
    MIN_PAIR_COUNT + counts.max() * 0.012,
    -0.85,
    f"selection threshold: {MIN_PAIR_COUNT} annotations",
    color=MUTED,
    fontsize=8,
    va="center",
    ha="left",
)

ax.set_yticks(ys)
ax.set_yticklabels(counts.index, fontsize=8)
for label, pair in zip(ax.get_yticklabels(), counts.index):
    if pair in selected:
        label.set_color(INK)
        label.set_fontweight("bold")

for pair in ("lw/cc", "sm/fc"):
    y = list(counts.index).index(pair)
    ax.annotate(
        "excluded: contains a syllable class",
        xy=(counts[pair] + counts.max() * 0.10, y),
        xytext=(counts.max() * 0.42, y),
        fontsize=7.5,
        color=CRITICAL,
        va="center",
        arrowprops=dict(arrowstyle="-", color=CRITICAL, linewidth=0.8),
    )

style_value_axis(ax, "x")
ax.set_xlabel("annotations")
save(
    fig,
    "annotations_per_pair",
    data={
        "par": counts.index,
        "anotaciones": counts,
        "seleccionada": [p in selected for p in counts.index],
        "umbral": MIN_PAIR_COUNT,
    },
)
plt.show()

In [ ]:
# Longest class on top, so both panels read in the same order.
order = experiment.groupby("pair").duration_s.median().sort_values(ascending=False).index.tolist()
positions = np.arange(1, len(order) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5.4), gridspec_kw={"wspace": 0.28})

# --- left: duration ---------------------------------------------------------
ax = axes[0]
data = [experiment.loc[experiment.pair == p, "duration_s"].values for p in order]
parts = ax.boxplot(
    data,
    orientation="horizontal",
    widths=0.55,
    patch_artist=True,
    showfliers=False,
    medianprops=dict(color=SURFACE, linewidth=1.6),
    whiskerprops=dict(color=BASELINE, linewidth=1),
    capprops=dict(color=BASELINE, linewidth=1),
)
for box in parts["boxes"]:
    box.set(facecolor=BLUE, edgecolor="none")

ax.set_xscale("log")
ax.set_ylim(len(order) + 0.7, 0.3)
for x, color, text in [
    (P.clip_len_s, MUTED, "3 s clip"),
    (2 * P.clip_len_s, CRITICAL, "6 s limit"),
]:
    ax.axvline(x, color=color, linewidth=1.1, linestyle=(0, (4, 3)), zorder=1)
    ax.text(
        x * 1.07,
        len(order) + 0.5,
        text,
        color=color,
        fontsize=8,
        va="bottom",
        ha="left",
        rotation=90,
    )

ax.set_yticks(positions, order)
style_value_axis(ax, "x")
ax.set_xlabel("event duration (s, log)")

# --- right: frequency band --------------------------------------------------
ax = axes[1]
band = (
    experiment.groupby("pair")
    .agg(
        lo=("low_freq_hz", lambda s: s.quantile(0.05)),
        hi=("high_freq_hz", lambda s: s.quantile(0.95)),
        lo_med=("low_freq_hz", "median"),
        hi_med=("high_freq_hz", "median"),
    )
    .loc[order]
)

for y, (_, row) in zip(positions, band.iterrows()):
    ax.plot(
        [row.lo, row.hi], [y, y], color=BLUE_FADED, linewidth=8, solid_capstyle="round", zorder=2
    )
    ax.plot(
        [row.lo_med, row.hi_med], [y, y], color=BLUE, linewidth=8, solid_capstyle="round", zorder=3
    )

ax.set_yticks(positions, order)
ax.set_ylim(len(order) + 0.7, 0.3)
ax.set_xlim(-400, P.f_max * 1.02)
ax.xaxis.set_major_formatter(lambda v, _: f"{v / 1000:g}k")
style_value_axis(ax, "x")
ax.set_xlabel("frequency (Hz)")
save(
    fig,
    "class_geometry",
    data={
        "clase": order,
        "duracion_s": {
            "q1": [float(np.percentile(d, 25)) for d in data],
            "mediana": [float(np.median(d)) for d in data],
            "q3": [float(np.percentile(d, 75)) for d in data],
        },
        "banda_hz": band.reset_index().to_dict(orient="list"),
    },
)
plt.show()

In [ ]:
cv = experiment.groupby("pair").bandwidth_hz.std() / experiment.groupby("pair").bandwidth_hz.mean()

N_COLS = 4
n_rows = int(np.ceil(len(CLASSES) / N_COLS))
last_row_cols = len(CLASSES) - (n_rows - 1) * N_COLS  # filled slots in the bottom row

fig, axes = plt.subplots(
    n_rows, N_COLS, figsize=(4.0 * N_COLS, 3.3 * n_rows), sharex=True, sharey=True
)
axes_flat = axes.ravel()
for ax, pair in zip(axes_flat, CLASSES):
    ax.scatter(
        experiment.duration_s, experiment.bandwidth_hz, s=4, color=GRID, edgecolors="none", zorder=1
    )
    sub = experiment[experiment.pair == pair]
    ax.scatter(
        sub.duration_s, sub.bandwidth_hz, s=8, color=BLUE, edgecolors="none", alpha=0.75, zorder=2
    )
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(pair, fontsize=11.5, pad=7)
    ax.annotate(
        f"n={len(sub):,} · bw CV {cv[pair]:.2f}",
        xy=(0.03, 0.05),
        xycoords="axes fraction",
        fontsize=9,
        color=INK_2,
    )
    ax.grid(True, color=GRID, linewidth=0.6)
    ax.set_axisbelow(True)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(BASELINE)

for ax in axes_flat[len(CLASSES) :]:
    ax.axis("off")

# Right-hand columns lose their bottom-row panel when CLASSES doesn't fill the
# grid exactly, so the last *populated* row per column carries the x label -
# for the short columns that's one row up, and sharex hides its ticks by default.
for col in range(N_COLS):
    bottom_row = n_rows - 1 if col < last_row_cols else n_rows - 2
    ax = axes[bottom_row, col]
    ax.tick_params(axis="x", labelbottom=True)
    ax.set_xlabel("duration (s)")
for row in range(n_rows):
    axes[row, 0].set_ylabel("bandwidth (Hz)")

fig.subplots_adjust(top=0.91, hspace=0.4, wspace=0.12)
save(fig, "class_geometry_facets")
plt.show()

In [ ]:
from data.manifest import build_manifest, split_manifest
from data.species import LabelSet

labels = LabelSet(CLASSES)
experiment["label"] = experiment.pair
manifest = build_manifest(experiment, labels)
splits = dict(
    zip(
        ("train", "val", "test"),
        split_manifest(manifest, seed=42, n_classes=len(labels), ratios=(0.6, 0.225, 0.175)),
    )
)

In [ ]:
boxes_per_split = (
    pd.DataFrame(
        {
            name: pd.Series([labels.name(int(c)) for w in split for c in w.labels]).value_counts()
            for name, split in splits.items()
        }
    )
    .fillna(0)
    .astype(int)
)
boxes_per_split["annotations"] = experiment.pair.value_counts()
boxes_per_split = boxes_per_split.sort_values("train", ascending=False)

In [ ]:
split_names = ["train", "val", "test"]
order = boxes_per_split.index.tolist()

fig, ax = plt.subplots(figsize=(9.5, 6.4))
peak = boxes_per_split[split_names].values.max()
ax.set_xlim(0, peak * 1.16)
ax.set_ylim(len(order) - 0.4, -0.7)
step = 0.26
for i, (name, color) in enumerate(zip(split_names, SERIES)):
    ys = np.arange(len(order)) + (i - 1) * step
    values = boxes_per_split.loc[order, name].values
    barh(ax, ys, values, color, height=step * 0.80)
    for y, v in zip(ys, values):
        ax.text(v + peak * 0.010, y, f"{v:,}", va="center", ha="left", fontsize=7.5, color=INK_2)

ax.set_yticks(np.arange(len(order)), order)
ax.set_ylim(len(order) - 0.4, -0.7)
style_value_axis(ax, "x")
ax.set_xlabel("boxes (one per window an event lands in)")

handles = [
    plt.Line2D([], [], marker="s", linestyle="", markersize=8, color=c, label=n)
    for c, n in zip(SERIES, split_names)
]
ax.legend(handles=handles, loc="upper left", bbox_to_anchor=(0.70, 0.34), ncol=1)
save(
    fig,
    "boxes_per_split",
    data=boxes_per_split.reset_index(names="clase"),
)
plt.show()

In [ ]:
per_window = np.array([len(w.boxes) for w in manifest])

fig, ax = plt.subplots(figsize=(8.6, 4.2))
values, edges = np.histogram(per_window, bins=np.arange(0.5, per_window.max() + 1.5))
centres = np.arange(1, per_window.max() + 1)
span = values.max()
ax.set_ylim(0, span * 1.16)
ax.set_xlim(0.3, per_window.max() + 0.7)
barv(ax, centres, values, BLUE)
for x, v in zip(centres, values):
    if v:
        ax.text(x, v + span * 0.02, f"{v:,}", ha="center", va="bottom", fontsize=7.5, color=INK_2)

ax.set_xticks(centres)
ax.grid(True, axis="y", color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_color(BASELINE)
ax.set_xlabel("boxes in one 3 s window")
ax.set_ylabel("windows")
save(
    fig,
    "boxes_per_window",
    data={"cajas_en_la_ventana": centres, "ventanas": values},
)
plt.show()

In [ ]:
from matplotlib.patches import FancyBboxPatch

from utils.audio import load_clip, mel_spectrogram, y_to_hz

busiest = max(splits["train"], key=lambda w: len(w.boxes))
mel = mel_spectrogram(P)(load_clip(busiest.audio_path, busiest.clip_start_s, P))
spectrogram = np.log(mel.numpy() + P.eps)

fig, ax = plt.subplots(figsize=(9, 4.4))
ax.imshow(
    spectrogram,
    origin="lower",
    aspect="auto",
    cmap="magma",
    extent=(0, P.clip_len_s, 0, 1),
    interpolation="nearest",
)

for (cx, cy, w, h), class_id in zip(busiest.boxes, busiest.labels):
    ax.add_patch(
        FancyBboxPatch(
            ((cx - w / 2) * P.clip_len_s, cy - h / 2),
            w * P.clip_len_s,
            h,
            boxstyle="round,pad=0,rounding_size=0.012",
            linewidth=1.6,
            edgecolor=SERIES[2],
            facecolor="none",
        )
    )
    ax.text(
        (cx - w / 2) * P.clip_len_s,
        cy + h / 2 + 0.012,
        labels.name(int(class_id)),
        color=SERIES[2],
        fontsize=8,
        va="bottom",
    )

ticks = np.linspace(0, 1, 6)
ax.set_yticks(ticks, [f"{y_to_hz(np.array([t]), P)[0] / 1000:.1f}k" for t in ticks])
ax.set_xlabel("time in clip (s)")
ax.set_ylabel("frequency (Hz, mel-spaced)")
for side in ("left", "bottom"):
    ax.spines[side].set_color(BASELINE)
save(fig, "window_example")
plt.show()